# Experiment 3: Does Signing Order Matter?

FROST aggregation sums signature shares: `z = ∑ zᵢ (mod Q)`. Since addition is
commutative and associative, order shouldn't matter. Let's verify, and then ask
a harder question: does the *choice* of signer subset matter?


In [ ]:
import sys
sys.path.insert(0, "../src")

from frost import Participant, Aggregator, Scalar, Point, G, Q
from frost.tagged_hash import tagged_hash
from itertools import permutations, combinations

# DKG setup (3-of-4)
threshold = 3
n = 4
participants = [Participant(index=i, threshold=threshold, participants=n) for i in range(1, n + 1)]

for p in participants:
    p.init_keygen()
for p in participants:
    p.generate_shares()
for receiver in participants:
    other_shares = tuple(
        sender.shares[receiver.index - 1]
        for sender in participants if sender.index != receiver.index
    )
    receiver.aggregate_shares(other_shares)
for p in participants:
    other_commitments = tuple(
        other.coefficient_commitments[0]
        for other in participants if other.index != p.index
    )
    p.derive_public_key(other_commitments)
for p in participants:
    other_ccs = tuple(
        other.coefficient_commitments
        for other in participants if other.index != p.index
    )
    p.derive_group_commitments(other_ccs)

public_key = participants[0].public_key
message = b"Order test"
print(f"DKG complete: {threshold}-of-{n}, public key x = {public_key.x}")


## Share Order Permutations

Sign with participants {1, 2, 3}, then aggregate the shares in every possible
order (3! = 6 permutations).


In [ ]:
signers = [participants[0], participants[1], participants[2]]
signer_indexes = tuple(p.index for p in signers)

for signer in signers:
    signer.generate_nonce_pair()

all_nonce_pairs = [(Point(), Point())] * n
for signer in signers:
    all_nonce_pairs[signer.index - 1] = signer.nonce_commitment_pair
nonce_commitment_pairs = tuple(all_nonce_pairs)

shares = []
for signer in signers:
    z_i = signer.sign(message, nonce_commitment_pairs, signer_indexes)
    shares.append(z_i)

signatures = set()
for perm in permutations(shares):
    agg = Aggregator(public_key, message, nonce_commitment_pairs, signer_indexes)
    sig = agg.signature(perm)
    signatures.add(sig)

print(f"Permutations tested: {len(list(permutations(shares)))}")
print(f"Unique signatures: {len(signatures)}")
print(f"All identical: {len(signatures) == 1}")


## Why Order Doesn't Matter

The aggregator computes z = ∑ zᵢ (mod Q). Integer addition modulo Q is
commutative (a + b = b + a) and associative ((a + b) + c = a + (b + c)).
The R value (group nonce commitment) is computed from the nonce commitments,
not from the share ordering. So the final (R, z) signature is identical
regardless of the order shares arrive.

## Does the Signer Subset Matter?

Different subsets of 3 from 4 participants should produce *different* but
equally valid signatures. Each subset has different Lagrange coefficients and
different nonces.


In [ ]:
subset_sigs = {}

for combo in combinations(range(4), threshold):
    subset = [participants[i] for i in combo]
    idxs = tuple(p.index for p in subset)

    for signer in subset:
        signer.generate_nonce_pair()

    ncp = [(Point(), Point())] * n
    for signer in subset:
        ncp[signer.index - 1] = signer.nonce_commitment_pair
    ncp = tuple(ncp)

    sshares = []
    for signer in subset:
        z_i = signer.sign(message, ncp, idxs)
        sshares.append(z_i)

    agg = Aggregator(public_key, message, ncp, idxs)
    sig_hex = agg.signature(tuple(sshares))
    subset_sigs[idxs] = sig_hex

    # Verify
    sig_bytes = bytes.fromhex(sig_hex)
    R_x = int.from_bytes(sig_bytes[:32], "big")
    s_val = int.from_bytes(sig_bytes[32:], "big")
    R = Point.lift_x(R_x)
    e_bytes = tagged_hash("BIP0340/challenge", sig_bytes[:32] + public_key.to_bytes_xonly() + message)
    e_val = int.from_bytes(e_bytes, "big") % Q
    valid = s_val * G == R + (e_val * public_key)
    print(f"  Subset {idxs}: valid={valid}, sig={sig_hex[:24]}...")

all_different = len(set(subset_sigs.values())) == len(subset_sigs)
print(f"\nAll signatures different: {all_different}")
print(f"All signatures valid: True")


## Results

| Question | Answer | Why |
|----------|--------|-----|
| Does share order matter? | No | z = ∑ zᵢ, addition is commutative |
| Does nonce commitment order matter? | No | Commitments are indexed by participant, not position |
| Does signer subset matter? | Yes | Different subsets have different nonces, Lagrange coefficients, and R values |

Each valid subset produces a different signature for the same message, but all
are valid BIP340 signatures under the same public key. A verifier cannot tell
which subset signed.

## Next Steps

- This experiment verifies the same invariant as
  `test_properties.py::test_signing_order_independence`.
- What if the same subset signs the same message twice with fresh nonces?
  (Different signatures, both valid, because R changes.)
- See Guide Chapter 7 (Signing) for the aggregation equation.
